# Checkpoint C3: Evaluator.py — Tu danh gia
Ket qua chay evaluator.py voi 12 cau hoi test.

In [ ]:
import csv
import json
from pathlib import Path

print('Imports OK: csv, json, pathlib')

In [ ]:
# Load test questions
questions_path = Path('../../synthetic-data/test-questions.csv')

questions = []
with open(questions_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        questions.append(row)

print(f'Loaded {len(questions)} test questions from {questions_path.name}')
print(f'Columns: {list(questions[0].keys()) if questions else "(empty)"}')

In [ ]:
# Print all 12 questions in a table
print(f'{"ID":<8} {"Category":<20} {"Question"}')
print('=' * 90)
for q in questions:
    qid = q.get('question_id', '?')
    cat = q.get('category', '?')
    question = q.get('question', '?')
    print(f'{qid:<8} {cat:<20} {question}')

print(f'\nTotal: {len(questions)} questions')

# Count by category
from collections import Counter
cat_counts = Counter(q.get('category', '') for q in questions)
print('\nBy category:')
for cat, count in cat_counts.items():
    print(f'  {cat}: {count}')

In [ ]:
def mock_evaluate(question_row, answer_text):
    """
    Mock evaluation function.
    Checks if the answer matches expected behavior for the category.
    
    Categories:
      - in-scope-direct / in-scope-cross: expect answer with citation
      - ambiguous: expect clarification request or cautious answer
      - out-of-scope: expect refusal + suggestion to contact HR
    """
    category = question_row.get('category', '')
    expected_behavior = question_row.get('expected_behavior', '')
    
    result = {
        'question_id': question_row.get('question_id', ''),
        'category': category,
        'answer': answer_text,
        'checks': {},
        'pass': True,
        'reason': ''
    }
    
    if 'out-of-scope' in category:
        # Expect refusal + suggestion
        has_refusal = any(kw in answer_text.lower() for kw in [
            'khong co', 'khong nam', 'ngoai pham vi', 'khong the tra loi',
            'toi khong co', 'lien he nhan su', 'nhan su'
        ])
        result['checks']['has_refusal'] = has_refusal
        if not has_refusal:
            result['pass'] = False
            result['reason'] = 'Missing refusal or HR contact suggestion'
    
    elif 'ambiguous' in category:
        # Expect clarification request
        has_clarification = any(kw in answer_text.lower() for kw in [
            'lam ro', 'cu the hon', 'xac dinh', 'phan loai',
            'vui long', 'khong ro'
        ])
        result['checks']['has_clarification'] = has_clarification
        if not has_clarification:
            result['pass'] = False
            result['reason'] = 'Missing clarification request'
    
    else:
        # in-scope: expect answer with citation
        has_content = len(answer_text) > 50
        has_citation = any(kw in answer_text for kw in [
            'POL-', 'muc', 'dieu', 'theo chinh sach', 'quy dinh',
            'trich dan', 'nguon:'
        ])
        result['checks']['has_content'] = has_content
        result['checks']['has_citation'] = has_citation
        if not has_content or not has_citation:
            result['pass'] = False
            missing = []
            if not has_content:
                missing.append('substantial content')
            if not has_citation:
                missing.append('citation')
            result['reason'] = f'Missing: {", ".join(missing)}'
    
    return result

print('mock_evaluate() function defined.')
print('Evaluates answers based on category:')
print('  in-scope-*     → expect answer with citation')
print('  ambiguous      → expect clarification request')
print('  out-of-scope   → expect refusal + HR contact')

In [ ]:
def cross_check_citation(quote, source_text):
    """
    Verify that a quote/citation actually exists in the source text.
    Uses fuzzy matching: check if key phrases from the quote appear in source.
    """
    if not quote or not source_text:
        return False, 'Empty quote or source'
    
    # Extract key phrases (sequences of 2+ words that are alphanumeric)
    import re
    phrases = re.findall(r'[\w\s]{4,}', quote)
    phrases = [p.strip() for p in phrases if len(p.strip()) > 3]
    
    if not phrases:
        return False, 'No checkable phrases found in quote'
    
    matched = 0
    for phrase in phrases[:5]:  # Check up to 5 phrases
        if phrase.lower() in source_text.lower():
            matched += 1
    
    ratio = matched / min(len(phrases), 5)
    
    if ratio >= 0.5:
        return True, f'{matched}/{min(len(phrases), 5)} phrases matched'
    else:
        return False, f'Only {matched}/{min(len(phrases), 5)} phrases matched (need >= 50%)'

# Quick test
ok, msg = cross_check_citation(
    'Nhan vien chinh thuc duoc 12 ngay phep nam',
    'Nhan vien chinh thuc duoc huong ngay phep nam theo tham nien: Duoi 5 nam: 12 ngay'
)
print(f'cross_check_citation test: {"PASS" if ok else "FAIL"} — {msg}')
print('\ncross_check_citation() function defined.')

In [ ]:
# Generate mock answers and run evaluation on all 12 questions

# Mock answer generator based on expected behavior
def generate_mock_answer(question_row):
    """Generate a mock answer that should pass evaluation."""
    category = question_row.get('category', '')
    q = question_row.get('question', '')
    source = question_row.get('expected_source', '')
    behavior = question_row.get('expected_behavior', '')
    
    if 'out-of-scope' in category:
        return (
            f"Xin loi, cau hoi \"{q}\" khong nam trong pham vi chinh sach nhan su "
            f"ma toi co the tra loi. Vui long lien he Phong Nhan su de duoc ho tro."
        )
    elif 'ambiguous' in category:
        return (
            f"Cau hoi \"{q}\" chua duoc cu the. "
            f"Vui long lam ro hon ve loai nghiep vu ban quan tam "
            f"de toi co the tra loi chinh xac."
        )
    else:
        return (
            f"Theo {source}: {behavior}. "
            f"Day la thong tin duoc trich dan tu van ban chinh sach cua cong ty. "
            f"Vui long tham khao van ban goc de biet them chi tiet."
        )

# Run evaluation
results = []
print(f'{"ID":<8} {"Category":<20} {"Result":<8} {"Reason"}')
print('=' * 80)

for q in questions:
    answer = generate_mock_answer(q)
    eval_result = mock_evaluate(q, answer)
    results.append(eval_result)
    
    status = 'PASS' if eval_result['pass'] else 'FAIL'
    reason = eval_result.get('reason', '')
    print(f'{eval_result["question_id"]:<8} {eval_result["category"]:<20} {status:<8} {reason}')

pass_count = sum(1 for r in results if r['pass'])
print(f'\nTotal: {pass_count}/{len(results)} PASSED')

In [ ]:
# SLI Summary
in_scope_results = [r for r in results if 'in-scope' in r['category']]
out_scope_results = [r for r in results if 'out-of-scope' in r['category']]
ambiguous_results = [r for r in results if 'ambiguous' in r['category']]

# In-scope accuracy
in_scope_pass = sum(1 for r in in_scope_results if r['pass'])
in_scope_accuracy = in_scope_pass / len(in_scope_results) if in_scope_results else 0

# Out-of-scope refusal rate
out_scope_pass = sum(1 for r in out_scope_results if r['pass'])
out_scope_refusal_rate = out_scope_pass / len(out_scope_results) if out_scope_results else 0

# Citation rate (in-scope answers that have citation)
cited_count = sum(1 for r in in_scope_results if r['checks'].get('has_citation', False))
citation_rate = cited_count / len(in_scope_results) if in_scope_results else 0

# Ambiguous handling rate
ambiguous_pass = sum(1 for r in ambiguous_results if r['pass'])
ambiguous_rate = ambiguous_pass / len(ambiguous_results) if ambiguous_results else 0

print('=' * 60)
print('SLI (Service Level Indicator) Summary')
print('=' * 60)
print(f'In-scope accuracy:        {in_scope_accuracy:.0%}  ({in_scope_pass}/{len(in_scope_results)} questions)')
print(f'Out-of-scope refusal:     {out_scope_refusal_rate:.0%}  ({out_scope_pass}/{len(out_scope_results)} questions)')
print(f'Citation rate:            {citation_rate:.0%}  ({cited_count}/{len(in_scope_results)} questions)')
print(f'Ambiguous handling:       {ambiguous_rate:.0%}  ({ambiguous_pass}/{len(ambiguous_results)} questions)')
print()
print('SLO Targets:')
print('  In-scope accuracy:     >= 90%')
print('  Out-of-scope refusal:  >= 95%')
print('  Citation rate:         >= 90%')

In [ ]:
print('=' * 60)
print('CHECKPOINT C3 SUMMARY')
print('=' * 60)
print()
print('Evaluator hoat dong voi mock answers.')
print()
print('Cac chuc nang da kiem tra:')
print('  1. Load test questions tu CSV')
print('  2. Phan loai theo category (in-scope, out-of-scope, ambiguous)')
print('  3. Mock evaluation: kiem tra answer vs expected behavior')
print('  4. Cross-check citation: xac minh trich dan')
print('  5. SLI metrics: accuracy, refusal rate, citation rate')
print()
print('De chay thuc te, can ket noi Gemini API.')
print('  - Thay generate_mock_answer() bang Gemini RAG pipeline')
print('  - Them API key: export GEMINI_API_KEY=your_key')
print()
print('NEXT: Chay checkpoint-step-d1.ipynb de chay danh gia day du.')